# Import libs & Data

In [1]:
import polars as pl
import polars.selectors as cs


In [2]:
# inference_res = pl.read_parquet(
#     "../../results/inference/pipeline_results_20251006_144126.parquet"
# )

res_df = pl.read_csv("../../results/evaluation/veatic_per_video_20251006_144126.csv")

In [3]:
len(res_df.columns)

38

In [4]:
mae_df = (
    res_df.select("video_id", "Stablization", cs.starts_with("mae"))
    .unpivot(
        index=["video_id", "Stablization"],
        variable_name="metrics",
        value_name="mae",
        on=cs.starts_with("mae"),
    )
    .sort("video_id")
)

with pl.Config(set_tbl_rows=20):
    display(mae_df.head(20))

video_id,Stablization,metrics,mae
i64,bool,str,f64
0,false,"""mae_scene_valence""",0.0068
0,true,"""mae_scene_valence""",0.017317
0,false,"""mae_scene_arousal""",0.230319
0,true,"""mae_scene_arousal""",0.243394
0,false,"""mae_face_valence""",0.037123
0,true,"""mae_face_valence""",0.036581
0,false,"""mae_face_arousal""",0.201124
0,true,"""mae_face_arousal""",0.204232
0,false,"""mae_fusion_valence""",0.02229


In [5]:
with pl.Config(set_tbl_rows=20):
    display(
        mae_df.group_by(["metrics", "Stablization"])
        .agg(pl.mean("mae").alias("mae_mean"))
        .sort(["metrics", "Stablization"])
        .pivot(index="metrics", on="Stablization", values="mae_mean")
    )

metrics,false,true
str,f64,f64
"""mae_face_arousal""",0.167998,0.168111
"""mae_face_valence""",0.208533,0.208359
"""mae_fusion_arousal""",0.160549,0.160829
"""mae_fusion_valence""",0.193418,0.19286
"""mae_scene_arousal""",0.203031,0.203255
"""mae_scene_valence""",0.239088,0.238353


# Potential way to compare results

The task is to select a sample of videos for the front-end. It's not a random selection, so I'm thinking how should we go about segmenting the videos? We can choose 3? Where? 

- Stabilizer: Enabled vs Disabled. Even though Fusion Mae is the same, but what about the viewer's perspective? Maybe across a sample of 10 videos, is the music tune better with or without stabilizer? 
- face_coverage: when `face_coverage` is high and `face_coverage` is low, does that make a difference? 

# Stabilizer: Enabled vs Disabled.

In [6]:
res_df

video_id,Stablization,gt_mean_valence,gt_mean_arousal,mean_scene_valence,mean_scene_arousal,coverage_scene,mae_scene_valence,mae_scene_arousal,var_scene_valence,var_scene_arousal,mean_face_valence,mean_face_arousal,coverage_face,mae_face_valence,mae_face_arousal,var_face_valence,var_face_arousal,mean_fusion_valence,mean_fusion_arousal,coverage_fusion,mae_fusion_valence,mae_fusion_arousal,var_fusion_valence,var_fusion_arousal,valence_std_dev,arousal_std_dev,face_beats_fusion_valence,scene_beats_fusion_valence,face_beats_fusion_arousal,scene_beats_fusion_arousal,delta_mae_scene_valence,delta_mae_scene_arousal,delta_mae_face_valence,delta_mae_face_arousal,delta_mae_fusion_valence,delta_mae_fusion_arousal,export_timestamp
i64,bool,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,f64,f64,f64,f64,f64,f64,str
0,false,-0.063334,-0.258307,-0.056534,-0.027988,null,0.0068,0.230319,0.001109,0.000217,-0.100457,-0.057183,0.87931,0.037123,0.201124,0.00049,0.000518,-0.085623,-0.068438,null,0.02229,0.189869,0.000382,0.000101,0.019545,0.01005,false,true,false,false,0.010517,0.013076,-0.000542,0.003108,0.015008,0.016291,"""20251006_144126"""
0,true,-0.063334,-0.258307,-0.08065,-0.014912,null,0.017317,0.243394,0.001324,0.000255,-0.099914,-0.054075,0.87931,0.036581,0.204232,0.000484,0.000545,-0.100631,-0.052147,null,0.037298,0.20616,0.000415,0.000128,0.020377,0.011314,true,true,true,false,0.010517,0.013076,-0.000542,0.003108,0.015008,0.016291,"""20251006_144126"""
1,false,-0.003551,0.137104,0.069304,-0.031735,null,0.072854,0.16884,0.001014,0.000197,-0.223129,0.274849,0.8,0.219579,0.137744,0.000464,0.000346,-0.123036,0.061729,null,0.119486,0.075375,0.000347,0.000115,0.018638,0.010737,false,true,false,false,0.007717,-0.016355,0.000143,0.001925,-0.025176,-0.009981,"""20251006_144126"""
1,true,-0.003551,0.137104,0.077021,-0.01538,null,0.080572,0.152485,0.000919,0.000198,-0.223273,0.276774,0.8,0.219722,0.139669,0.000452,0.00035,-0.09786,0.07171,null,0.09431,0.065394,0.000416,0.000122,0.020397,0.011023,false,true,false,false,0.007717,-0.016355,0.000143,0.001925,-0.025176,-0.009981,"""20251006_144126"""
10,false,-0.384963,0.146455,-0.065421,-0.017318,null,0.319542,0.163773,0.000982,0.000251,-0.166162,0.300696,0.955882,0.218801,0.154241,0.00022,0.000245,-0.166006,0.155372,null,0.218957,0.008917,0.00016,0.000062,0.012665,0.007868,true,false,false,false,0.021704,-0.003523,-0.001252,0.000464,0.015137,0.020322,"""20251006_144126"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
97,true,0.2349,-0.047971,0.096364,-0.025462,null,0.138536,0.022509,0.001492,0.000183,-0.155894,0.20391,0.80597,0.390794,0.251881,0.000461,0.000364,-0.026673,0.037739,null,0.261573,0.08571,0.000488,0.000078,0.022096,0.008834,false,true,false,true,0.036924,0.013216,-0.000534,0.001015,0.033614,0.001661,"""20251006_144126"""
98,false,0.143648,0.069885,0.133924,-0.061393,null,0.009725,0.131278,0.000466,0.000046,-0.195472,0.155017,1.0,0.33912,0.085132,0.000194,0.000174,-0.085194,-0.011945,null,0.228842,0.081831,0.000088,0.000019,0.009372,0.004397,false,true,false,false,0.020048,-0.001897,0.000904,-0.000156,-0.003265,-0.008395,"""20251006_144126"""
98,true,0.143648,0.069885,0.113876,-0.059496,null,0.029772,0.129381,0.0005,0.000055,-0.196375,0.154861,1.0,0.340023,0.084976,0.000196,0.000176,-0.081929,-0.003551,null,0.225577,0.073436,0.00009,0.000021,0.00949,0.004632,false,true,false,false,0.020048,-0.001897,0.000904,-0.000156,-0.003265,-0.008395,"""20251006_144126"""


# Video Selection: Lowest Valence MAE

In [7]:
face_valence_top = (
    res_df.filter(
        (pl.col("mae_face_valence") < pl.col("mae_scene_valence"))
        & (pl.col("mae_face_valence") < pl.col("mae_fusion_valence"))
    )
    .sort("mae_face_valence")
    .unique(subset=["video_id"], keep="first")
    .sort("mae_face_valence")
    .select(
        "video_id",
        "Stablization",
        "mae_face_valence",
        "mae_scene_valence",
        "mae_fusion_valence",
    )
    .with_columns(cs.starts_with("mae_").round(4))
    .head(3)
)

scene_valence_top = (
    res_df.filter(
        (pl.col("mae_scene_valence") < pl.col("mae_face_valence"))
        & (pl.col("mae_scene_valence") < pl.col("mae_fusion_valence"))
    )
    .sort("mae_scene_valence")
    .unique(subset=["video_id"], keep="first")
    .sort("mae_scene_valence")
    .select(
        "video_id",
        "Stablization",
        "mae_scene_valence",
        "mae_face_valence",
        "mae_fusion_valence",
    )
    .with_columns(cs.starts_with("mae_").round(4))
    .head(3)
)

fusion_valence_top = (
    res_df.filter(
        (pl.col("mae_fusion_valence") < pl.col("mae_scene_valence"))
        & (pl.col("mae_fusion_valence") < pl.col("mae_face_valence"))
    )
    .sort("mae_fusion_valence")
    .unique(subset=["video_id"], keep="first")
    .sort("mae_fusion_valence")
    .select(
        "video_id",
        "Stablization",
        "mae_fusion_valence",
        "mae_scene_valence",
        "mae_face_valence",
    )
    .with_columns(cs.starts_with("mae_").round(4))
    .head(3)
)

with pl.Config(set_tbl_rows=10):
    print("3 videos with lowest face_valence MAE"), display(face_valence_top)
    print("3 videos with lowest scene_valence MAE"), display(scene_valence_top)
    print("3 videos with lowest fusion_valence MAE"), display(fusion_valence_top)


3 videos with lowest face_valence MAE


video_id,Stablization,mae_face_valence,mae_scene_valence,mae_fusion_valence
i64,bool,f64,f64,f64
42,true,0.0053,0.236,0.0683
46,false,0.0106,0.0463,0.0107
53,false,0.0124,0.0179,0.0197


3 videos with lowest scene_valence MAE


video_id,Stablization,mae_scene_valence,mae_face_valence,mae_fusion_valence
i64,bool,f64,f64,f64
120,true,0.0022,0.189,0.1221
86,true,0.0058,0.2443,0.0768
0,false,0.0068,0.0371,0.0223


3 videos with lowest fusion_valence MAE


video_id,Stablization,mae_fusion_valence,mae_scene_valence,mae_face_valence
i64,bool,f64,f64,f64
40,true,0.0002,0.1632,0.0921
49,false,0.0005,0.0123,0.0238
51,false,0.0016,0.2549,0.1073


In [8]:
face_arousal_top = (
    res_df.filter(
        (pl.col("mae_face_arousal") < pl.col("mae_scene_arousal"))
        & (pl.col("mae_face_arousal") < pl.col("mae_fusion_arousal"))
    )
    .sort("mae_face_arousal")
    .unique(subset=["video_id"], keep="first")
    .sort("mae_face_arousal")
    .select(
        "video_id",
        "Stablization",
        "mae_face_arousal",
        "mae_scene_arousal",
        "mae_fusion_arousal",
    )
    .with_columns(cs.starts_with("mae_").round(4))
    .head(3)
)

scene_arousal_top = (
    res_df.filter(
        (pl.col("mae_scene_arousal") < pl.col("mae_face_arousal"))
        & (pl.col("mae_scene_arousal") < pl.col("mae_fusion_arousal"))
    )
    .sort("mae_scene_arousal")
    .unique(subset=["video_id"], keep="first")
    .sort("mae_scene_arousal")
    .select(
        "video_id",
        "Stablization",
        "mae_scene_arousal",
        "mae_face_arousal",
        "mae_fusion_arousal",
    )
    .with_columns(cs.starts_with("mae_").round(4))
    .head(3)
)

fusion_arousal_top = (
    res_df.filter(
        (pl.col("mae_fusion_arousal") < pl.col("mae_scene_arousal"))
        & (pl.col("mae_fusion_arousal") < pl.col("mae_face_arousal"))
    )
    .sort("mae_fusion_arousal")
    .unique(subset=["video_id"], keep="first")
    .sort("mae_fusion_arousal")
    .select(
        "video_id",
        "Stablization",
        "mae_fusion_arousal",
        "mae_scene_arousal",
        "mae_face_arousal",
    )
    .with_columns(cs.starts_with("mae_").round(4))
    .head(3)
)

with pl.Config(set_tbl_rows=10):
    print("3 videos with lowest face_arousal MAE"), display(face_arousal_top)
    print("3 videos with lowest scene_arousal MAE"), display(scene_arousal_top)
    print("3 videos with lowest fusion_arousal MAE"), display(fusion_arousal_top)


3 videos with lowest face_arousal MAE


video_id,Stablization,mae_face_arousal,mae_scene_arousal,mae_fusion_arousal
i64,bool,f64,f64,f64
102,false,0.0015,0.2946,0.1972
95,false,0.0019,0.1836,0.1089
81,true,0.0039,0.131,0.0939


3 videos with lowest scene_arousal MAE


video_id,Stablization,mae_scene_arousal,mae_face_arousal,mae_fusion_arousal
i64,bool,f64,f64,f64
79,false,0.0007,0.2462,0.1131
34,true,0.002,0.229,0.0936
96,true,0.0036,0.2396,0.0934


3 videos with lowest fusion_arousal MAE


video_id,Stablization,mae_fusion_arousal,mae_scene_arousal,mae_face_arousal
i64,bool,f64,f64,f64
12,false,0.0032,0.0947,0.1181
55,false,0.0044,0.1068,0.1151
114,true,0.0071,0.0876,0.2166


# Check cols

Uncertainty presented here takes the square root of variance: Gives a standard deviation with the same units as valence/arousal (i.e., you can say “±0.1 valence”), which is easier to contextualize.

In [9]:
res_df.select("video_id", "Stablization", cs.ends_with("std_dev")).sort("video_id")

video_id,Stablization,valence_std_dev,arousal_std_dev
i64,bool,f64,f64
0,false,0.019545,0.01005
0,true,0.020377,0.011314
1,false,0.018638,0.010737
1,true,0.020397,0.011023
2,false,0.030047,0.014163
…,…,…,…
121,true,0.007766,0.003911
122,false,0.017297,0.007747
122,true,0.017629,0.008031


In [ ]:
(
    res_df.filter(
        pl.col("face_beats_fusion_valence") == False,
        pl.col("face_beats_fusion_arousal") == False,
        pl.col("scene_beats_fusion_valence") == False,
        pl.col("scene_beats_fusion_arousal") == False,
    )
    .select("video_id", "Stablization", cs.starts_with("mae"))
    .unpivot(
        on=cs.ends_with("valence"),
        index=["video_id", "Stablization"],
        variable_name="valence",
        value_name="mae",
    )
    .sort("video_id", "Stablization")
)

video_id,Stablization,valence,mae
i64,bool,str,f64
9,false,"""mae_scene_valence""",0.250187
9,false,"""mae_fusion_valence""",0.122814
9,false,"""mae_face_valence""",0.193488
9,true,"""mae_fusion_valence""",0.092993
9,true,"""mae_face_valence""",0.194132
…,…,…,…
115,false,"""mae_face_valence""",0.176396
115,false,"""mae_scene_valence""",0.097148
115,true,"""mae_face_valence""",0.175914


In [ ]:
(
    res_df.filter(pl.col("video_id") == 40)
    .select("video_id", cs.starts_with("mean"), cs.ends_with("std_dev"))
    .with_columns(pl.all().round(3))
    .glimpse()
)

Rows: 2
Columns: 9
$ video_id            <i64> 40, 40
$ mean_scene_valence  <f64> 0.118, 0.133
$ mean_scene_arousal  <f64> -0.061, -0.065
$ mean_face_valence   <f64> -0.123, -0.122
$ mean_face_arousal   <f64> 0.275, 0.273
$ mean_fusion_valence <f64> -0.059, -0.03
$ mean_fusion_arousal <f64> 0.043, 0.038
$ valence_std_dev     <f64> 0.013, 0.012
$ arousal_std_dev     <f64> 0.007, 0.006

